# 02 — Exploratory Data Analysis and source-data validation

In [1]:
from pathlib import Path
import pandas as pd, numpy as np
from IPython.display import display
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks': PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED = PROJECT_ROOT/'data'/'processed'
AUDIT = PROJECT_ROOT/'reports'/'data_quality'
FIGURES = PROJECT_ROOT/'reports'/'figures'/'final_eda'
HANDOFF = PROJECT_ROOT/'reports'/'modelling_handoff'
for d in (PROCESSED,AUDIT,FIGURES,HANDOFF): d.mkdir(parents=True,exist_ok=True)
print('Project:',PROJECT_ROOT)

from matplotlib import pyplot as plt
import matplotlib
matplotlib.use('Agg') # save figures without interactive-window dependencies
from matplotlib import pyplot as plt
from scipy.stats import spearmanr
LISTINGS_PATH=PROCESSED/'listings_master.parquet'
if not LISTINGS_PATH.exists(): raise FileNotFoundError(LISTINGS_PATH)
master=pd.read_parquet(LISTINGS_PATH)
required={'country','location','id','price','room_type'}
missing=required-set(master.columns)
if missing: raise ValueError('Required columns missing: '+str(sorted(missing)))
print('Listings:',len(master),'Markets:',master.location.nunique())

def savefig(fig,name):
    fig.tight_layout()
    fig.savefig(FIGURES/f'{name}.png',dpi=170,bbox_inches='tight')
    plt.close(fig)
    print('Saved',name)

def savetable(frame,name):
    frame.to_csv(AUDIT/f'{name}.csv',index=False)
    print('Saved',name)

Project: C:\Users\erins\OneDrive\Documents\Airbnb_VSCode_Project\Airbnb_EDACode_Project_\reordered notebooks
Listings: 756312 Markets: 45


## 1. Source integrity and missingness
Report duplicate identifiers, cross-location overlap and column-level/market-level missingness before transforming any data.

In [2]:
keys=['location','id']
null_key=master[keys].isna().any(axis=1)
local_duplicate=master.duplicated(keys,keep=False)
summary=pd.DataFrame([{'rows':len(master),'markets':master.location.nunique(),
  'countries':master.country.nunique(),'null_key_rows':int(null_key.sum()),
  'duplicate_key_rows':int(local_duplicate.sum()),
  'global_ids_in_multiple_locations':int(master.groupby('id')['location'].nunique().gt(1).sum())}])
display(summary);savetable(summary,'dataset_integrity_summary')
exceptions=master.loc[null_key|local_duplicate, [c for c in ['country','location','id','source_file'] if c in master]].head(10000)
savetable(exceptions,'duplicate_listing_key_examples')
overlap=master.groupby('id')['location'].nunique().rename('number_of_locations').reset_index()
savetable(overlap.loc[overlap.number_of_locations>1],'cross_location_listing_id_overlap')
miss=pd.DataFrame({'column':master.columns,'missing_n':master.isna().sum().values,
 'missing_pct':np.round(100*master.isna().mean().values,2),'dtype':master.dtypes.astype(str).values})
miss=miss.sort_values('missing_pct',ascending=False)
savetable(miss,'missingness_by_column');display(miss.head(20))
key_columns=[x for x in ['price','price_quote_price_per_night','accommodates','bedrooms','bathrooms_text','latitude','longitude','room_type'] if x in master]
market_missing=(master.groupby(['country','location'],dropna=False)[key_columns]
  .agg(lambda x:round(100*x.isna().mean(),2)).reset_index())
savetable(market_missing,'missingness_by_location_pct')

,rows,markets,countries,null_key_rows,duplicate_key_rows,global_ids_in_multiple_locations
0,756312,45,16,0,0,0


Saved dataset_integrity_summary
Saved duplicate_listing_key_examples
Saved cross_location_listing_id_overlap
Saved missingness_by_column


,column,missing_n,missing_pct,dtype
7,neighborhood_overview,756312,100.00,float64
27,host_neighbourhood,756312,100.00,float64
22,host_response_rate,756312,100.00,float64
23,host_acceptance_rate,756312,100.00,float64
14,host_since,756312,100.00,float64
21,host_response_time,756312,100.00,float64
25,host_thumbnail_url,756312,100.00,float64
30,host_verifications,756312,100.00,float64
84,instant_bookable,756312,100.00,float64
60,calendar_updated,756312,100.00,float64


Saved missingness_by_location_pct


## 2. Price definition and currency audit.

In [3]:
def parse_source_money(s):
    return pd.to_numeric(s.astype('string').str.strip()
        .str.replace(r'[^0-9.\-]','',regex=True),errors='coerce')
master['price_numeric']=parse_source_money(master['price'])
master['price_quote_numeric']=(parse_source_money(master['price_quote_price_per_night'])
  if 'price_quote_price_per_night' in master else np.nan)
master['price_missing']=master.price_numeric.isna()
master['price_nonpositive']=master.price_numeric.notna() & master.price_numeric.le(0)
market_price=(master.groupby(['country','location'],dropna=False)
 .agg(n=('id','size'),valid_price=('price_numeric',lambda x:int(x.gt(0).sum())),
 missing_prices=('price_missing','sum'),nonpositive=('price_nonpositive','sum'),
 median_price=('price_numeric','median'),p01=('price_numeric',lambda x:x.quantile(.01)),
 p99=('price_numeric',lambda x:x.quantile(.99)),quote_median=('price_quote_numeric','median'))
 .reset_index())
market_price['missing_price_pct']=(100*market_price.missing_prices/market_price.n).round(2)
market_price['quote_price_ratio']=market_price.quote_median/market_price.median_price.replace(0,np.nan)
savetable(market_price,'price_definition_market_audit');display(market_price)
# Source format is preserved; internal comparisons do not independently verify currency.
expected_currency={'Austria':'EUR','Belgium':'EUR','Czech Republic':'CZK','Denmark':'DKK',
'France':'EUR','Germany':'EUR','Greece':'EUR','Ireland':'EUR','Italy':'EUR',
'Latvia':'EUR','Netherlands':'EUR','Norway':'NOK','Portugal':'EUR','Spain':'EUR',
'Switzerland':'CHF','United Kingdom':'GBP'}
market_price['expected_local_currency']=market_price.country.map(expected_currency)
market_price['currency_status']='EXPECTED_NOT_SOURCE_VERIFIED'
market_price.loc[market_price.expected_local_currency.isna(),'currency_status']='UNKNOWN'
market_price.loc[market_price.location.isin(['Geneva','Vaud']),'currency_status']='SWISS_SCALE_UNRESOLVED'
savetable(market_price,'currency_and_price_quality_status')

Saved price_definition_market_audit


,country,location,n,valid_price,missing_prices,nonpositive,median_price,p01,p99,quote_median,missing_price_pct,quote_price_ratio
0,Austria,Vienna,13283,10492,2791,0,118.5,20.9746,592.18,118.5,21.01,1.0
1,Belgium,Antwerp,2535,2069,466,0,139.5,10.1596,784.7044,139.5,18.38,1.0
2,Belgium,Brussels,5078,4378,700,0,123.25,11.0885,730.7295,123.25,13.78,1.0
3,Belgium,Ghent,1383,1211,172,0,148.5,39.4,585.7,148.5,12.44,1.0
4,Czech Republic,Prague,11615,10583,1032,0,2473.64,515.0,15027.8694,2473.64,8.89,1.0
5,Denmark,Copenhagen,23144,13860,9284,0,1689.0,461.9031,6368.355,1689.0,40.11,1.0
6,France,Bordeaux,12217,8413,3804,0,107.79,17.8804,665.9796,107.79,31.14,1.0
7,France,Lyon,9317,5427,3890,0,103.68,24.84,568.37,103.68,41.75,1.0
8,France,Paris,77679,48402,29277,0,205.5,29.87,1874.98,205.5,37.69,1.0
9,Germany,Berlin,12776,8441,4335,0,130.67,9.208,632.3,130.67,33.93,1.0


Saved currency_and_price_quality_status


In [4]:
# Outliers and extreme markets: descriptive flags, no automatic trimming.
priced=master.loc[master.price_numeric.gt(0)].copy()
priced['market_p99']=priced.groupby('location')['price_numeric'].transform(lambda s:s.quantile(.99))
priced['above_market_p99']=priced.price_numeric.gt(priced.market_p99)
outliers=(priced.groupby(['country','location'],dropna=False)
 .agg(priced_rows=('id','size'),above_market_p99=('above_market_p99','sum'),
 median_price=('price_numeric','median'),max_price=('price_numeric','max')).reset_index())
savetable(outliers,'price_outlier_audit_by_market')

Saved price_outlier_audit_by_market


## 3. Supply, room types and host structure

In [5]:
counts=master.location.value_counts().rename('listings').reset_index();counts.columns=['location','listings']
display(counts.head(15));savetable(counts,'market_listing_counts')
fig,ax=plt.subplots(figsize=(11,7));t=counts.head(20).iloc[::-1]
ax.barh(t.location,t.listings);ax.set(xlabel='Listing records',title='20 largest Airbnb source markets');savefig(fig,'listing_counts_top20')
room_share=(master.groupby(['country','location','room_type'],dropna=False).size().rename('n').reset_index())
room_share['market_total']=room_share.groupby('location')['n'].transform('sum')
room_share['share_pct']=100*room_share.n/room_share.market_total
savetable(room_share,'room_types_by_market')
if 'host_id' in master:
    hosts=master.groupby(['country','location'],dropna=False).agg(listings=('id','size'),hosts=('host_id','nunique')).reset_index()
    savetable(hosts,'host_market_summary')

,location,listings
0,London,92638
1,Paris,77679
2,Sicily,56873
3,Puglia,48683
4,Rome,37084
5,South Aegean,36680
6,Crete,27333
7,Milan,25750
8,Lisbon,24876
9,Copenhagen,23144


Saved market_listing_counts
Saved listing_counts_top20
Saved room_types_by_market
Saved host_market_summary


## 4. Geographic coverage and neighborhood supply

In [6]:
for field in ['latitude','longitude']:
    if field in master: master[field]=pd.to_numeric(master[field],errors='coerce')
if {'latitude','longitude'}.issubset(master.columns):
    valid_geo=(master.latitude.between(-90,90)&master.longitude.between(-180,180)&
      ~((master.latitude==0)&(master.longitude==0)))
    geo_validation=pd.DataFrame([{'rows':len(master),'valid_coordinates':int(valid_geo.sum()),
         'invalid_or_missing_coordinates':int((~valid_geo).sum())}])
    display(geo_validation);savetable(geo_validation,'coordinate_quality')
else:valid_geo=pd.Series(False,index=master.index)
hood_col=next((c for c in ['neighbourhood_cleansed','neighbourhood'] if c in master),None)
if hood_col:
    hood=(master.loc[valid_geo].groupby(['country','location',hood_col],dropna=False)
       .agg(listings=('id','size'),hosts=('host_id','nunique') if 'host_id' in master else ('id','nunique'),
       latitude=('latitude','median'),longitude=('longitude','median')).reset_index())
    savetable(hood,'neighborhood_supply')
else:print('Neighborhood column absent; skipped neighborhood aggregation.')

,rows,valid_coordinates,invalid_or_missing_coordinates
0,756312,756312,0


Saved coordinate_quality
Saved neighborhood_supply


In [7]:
# Supplemental descriptive relationship: neighborhood supply vs local median asking price.
# Calculated here for EDA only; no persistent listing-level competitor features yet.
if hood_col:
    hood_summary=(master.groupby(['country','location',hood_col],dropna=False)
        .agg(listing_count=('id','size'),median_price=('price_numeric','median'),
             price_observations=('price_numeric','count')).reset_index())
    hood_summary=hood_summary.loc[(hood_summary.price_observations>=10)&(hood_summary.median_price>0)]
    savetable(hood_summary,'neighborhood_price_and_supply')
    markets=master.location.value_counts().head(9).index
    fig,axes=plt.subplots(3,3,figsize=(15,13));axes=axes.ravel()
    for i,market in enumerate(markets):
        part=hood_summary.loc[hood_summary.location.eq(market)]
        axes[i].scatter(part.listing_count,part.median_price,alpha=.7,s=20)
        axes[i].set_title(str(market));axes[i].set_xlabel('Neighborhood listing count')
        axes[i].set_ylabel('Neighborhood median local price')
    for ax in axes[len(markets):]:ax.axis('off')
    savefig(fig,'price_vs_neighborhood_supply_market_panels')


Saved neighborhood_price_and_supply
Saved price_vs_neighborhood_supply_market_panels


In [8]:
# Save the Europe neighborhood-supply map as HTML (optional package).
# This is a descriptive symbol map, NOT listings-per-km² polygon density.
if hood_col and {'latitude','longitude'}.issubset(master.columns):
    try:
        import plotly.express as px
        geo_hood=(master.loc[valid_geo].groupby(['country','location',hood_col],dropna=False)
            .agg(listing_count=('id','size'),latitude=('latitude','median'),
                 longitude=('longitude','median')).reset_index())
        fig=px.scatter_geo(geo_hood,lat='latitude',lon='longitude',size='listing_count',
           color='listing_count',hover_name=hood_col,hover_data=['country','location'],
           scope='europe',projection='natural earth',title='Airbnb neighborhood listing supply across Europe')
        map_dir=PROJECT_ROOT/'reports'/'maps';map_dir.mkdir(parents=True,exist_ok=True)
        fig.write_html(str(map_dir/'neighborhood_supply_europe.html'),include_plotlyjs=True)
        print('Saved interactive map:',map_dir/'neighborhood_supply_europe.html')
    except ImportError:
        print('Optional plotly unavailable; skipped HTML map.')


Saved interactive map: C:\Users\erins\OneDrive\Documents\Airbnb_VSCode_Project\Airbnb_EDACode_Project_\reordered notebooks\reports\maps\neighborhood_supply_europe.html


## 5. Reviews and seasonality

In [9]:
import duckdb
REVIEWS_PATH=PROCESSED/'reviews_clean.parquet'
if not REVIEWS_PATH.is_file():
    raise FileNotFoundError('Expected prepared reviews_clean.parquet: '+str(REVIEWS_PATH))
con=duckdb.connect()
con.execute("SET memory_limit='12GB'");con.execute('SET threads=4')
q=str(REVIEWS_PATH).replace("'","''")
con.execute(f"CREATE VIEW reviews AS SELECT * FROM read_parquet('{q}')")
review_summary=con.execute('''SELECT country,location,COUNT(*) AS review_count,
 COUNT(DISTINCT listing_id) AS reviewed_listings,
 MIN(TRY_CAST(date AS DATE)) AS first_review,
 MAX(TRY_CAST(date AS DATE)) AS latest_review
 FROM reviews GROUP BY country,location ORDER BY review_count DESC''').df()
display(review_summary.head(15));savetable(review_summary,'review_activity_by_market')
# Full-history seasonal distributions describe historic activity, not occupancy.
review_seasons=con.execute('''SELECT location,country,YEAR(TRY_CAST(date AS DATE)) AS year,
 CASE WHEN MONTH(TRY_CAST(date AS DATE)) IN (12,1,2) THEN 'Winter'
 WHEN MONTH(TRY_CAST(date AS DATE)) IN (3,4,5) THEN 'Spring'
 WHEN MONTH(TRY_CAST(date AS DATE)) IN (6,7,8) THEN 'Summer'
 ELSE 'Autumn' END AS season,COUNT(*) AS reviews
 FROM reviews WHERE TRY_CAST(date AS DATE) IS NOT NULL
 GROUP BY 1,2,3,4''').df()
savetable(review_seasons,'review_seasonality')
con.close()

,country,location,review_count,reviewed_listings,first_review,latest_review
0,Italy,Rome,2477671,32827,2010-05-14,2026-07-06
1,France,Paris,2404369,61781,2010-01-26,2026-06-28
2,United Kingdom,London,2241353,70832,2009-12-21,2026-06-30
3,Portugal,Lisbon,1886932,21794,2010-07-24,2026-07-02
4,Spain,Madrid,1419360,19105,2010-07-06,2026-07-01
5,Italy,Sicily,1301042,40373,2009-08-26,2026-07-02
6,Italy,Milan,1148169,21941,2010-12-20,2026-07-02
7,Portugal,Porto,1131143,13683,2011-05-10,2026-07-02
8,Italy,Florence,1108421,12216,2010-06-17,2026-07-02
9,Spain,Barcelona,1033523,11822,2010-10-03,2026-07-02


Saved review_activity_by_market
Saved review_seasonality


## 6. Between-variable EDA — all analysis before feature engineering
To avoid mixing currencies, most plots use **within-market panels** or percentage-normalized descriptive quantities. The results are observational associations, not causal effects.

In [10]:
# Nightly price vs room type, by location (each location keeps its own price units)
top=priced.location.value_counts().head(12).index
agg=(priced.loc[priced.location.isin(top)].groupby(['location','room_type'],dropna=False)
 .agg(median_price=('price_numeric','median'),n=('id','size')).reset_index())
agg=agg.loc[agg.n>=10]
fig,axes=plt.subplots(4,3,figsize=(17,15));axes=axes.ravel()
for i,loc in enumerate(top):
    t=agg.loc[agg.location.eq(loc)].sort_values('median_price')
    if len(t):axes[i].barh(t.room_type.astype(str),t.median_price)
    axes[i].set_title(loc);axes[i].set_xlabel('Local nominal nightly price')
for ax in axes[len(top):]:ax.axis('off')
savefig(fig,'price_by_room_type_market_panels')

Saved price_by_room_type_market_panels


In [11]:
# Price vs capacity and bedrooms, per-market panels
for col,name,cap in [('accommodates','price_vs_capacity',12),('bedrooms','price_vs_bedrooms',8)]:
    if col not in priced:
        print('Skipping',name,'column not found');continue
    v=priced[['location','price_numeric',col]].copy()
    v['x']=pd.to_numeric(v[col],errors='coerce');v=v.loc[v.x.between(1,cap)]
    s=(v.groupby(['location','x']).agg(median=('price_numeric','median'),n=('price_numeric','size')).reset_index())
    s=s.loc[s.n>=10]
    top_loc=priced.location.value_counts().head(12).index
    fig,axes=plt.subplots(4,3,figsize=(16,13));axes=axes.ravel()
    for i,loc in enumerate(top_loc):
        t=s.loc[s.location.eq(loc)].sort_values('x')
        axes[i].plot(t.x,t['median'],marker='o');axes[i].set_title(loc)
        axes[i].set_xlabel(col);axes[i].set_ylabel('Median local price')
    for ax in axes[len(top_loc):]:ax.axis('off')
    savefig(fig,name)

Saved price_vs_capacity
Saved price_vs_bedrooms


In [12]:
# Price vs availability and cumulative reviews; within-location correlations.
rows=[]
for col in ['availability_365','number_of_reviews']:
    if col not in priced:continue
    temp=priced[['location','price_numeric',col]].copy()
    temp[col]=pd.to_numeric(temp[col],errors='coerce')
    for loc,g in temp.dropna().groupby('location'):
        if len(g)<30 or g[col].nunique()<2 or g.price_numeric.nunique()<2:continue
        rho,p=spearmanr(g[col],g.price_numeric)
        rows.append({'location':loc,'variable':col,'rho':rho,'p_value':p,'n':len(g)})
correlations=pd.DataFrame(rows,columns=['location','variable','rho','p_value','n'])
savetable(correlations,'price_relationship_correlations')
if len(correlations):
    for var,g in correlations.groupby('variable'):
        t=g.sort_values('rho');fig,ax=plt.subplots(figsize=(10,9))
        ax.barh(t.location,t.rho);ax.axvline(0,color='black',linewidth=.7)
        ax.set(xlabel='Spearman correlation with local price',title=f'Price relationship: {var}')
        savefig(fig,'correlation_'+var)

Saved price_relationship_correlations
Saved correlation_availability_365
Saved correlation_number_of_reviews


In [13]:
# Exploratory country summaries of review and supply activity. Do not label as occupancy.
if 'review_summary' in globals():
    merged=(counts.merge(review_summary[['location','review_count']],on='location',how='left'))
    merged['reviews_per_listing_historical']=merged.review_count/merged.listings
    savetable(merged,'historical_reviews_per_listing_by_market')
print('EDA finished. All figures in:',FIGURES)
print('Currency units remain unverified: no pooled European price chart or model approved.')

Saved historical_reviews_per_listing_by_market
EDA finished. All figures in: C:\Users\erins\OneDrive\Documents\Airbnb_VSCode_Project\Airbnb_EDACode_Project_\reordered notebooks\reports\figures\final_eda
Currency units remain unverified: no pooled European price chart or model approved.
